# Clase 6 — LLMs (GPT, Llama) y tu primer chatbot con Hugging Face

## Objetivos de aprendizaje

- Explicar qué es un LLM (Large Language Model) en términos simples.
- Conocer, a alto nivel, las características de la familia Llama 2 y por qué
  el código de ejemplo de su PDF oficial es ficticio.
- Distinguir un modelo seq2seq (encoder-decoder) de un LLM causal
  instruction-tuned.
- Construir y ejecutar dos chatbots de terminal 100% gratuitos con Hugging
  Face Transformers: `facebook/blenderbot-400M-distill` y
  `HuggingFaceTB/SmolLM2-360M-Instruct`.

## Qué vamos a construir

Dos chatbots de terminal que corren en la CPU gratuita de Colab, sin ninguna
API key. Vas a poder hablar con ambos y comparar cómo responden.

## Apunte rápido: ¿qué es un LLM?

Un LLM es un modelo de deep learning entrenado con muchísimo texto, capaz de
generar, traducir, resumir o analizar lenguaje (definición del glosario
oficial del curso). GPT (OpenAI) y Llama (Meta AI) son dos familias muy
conocidas de LLM, ambas basadas en la arquitectura Transformer y en el
mecanismo de atención que viste en la clase anterior — pero a una escala
mucho mayor (miles de millones de parámetros).

**Aviso importante**: el PDF oficial "Introduction to Meta Llama 2" incluye
un ejemplo de código con `from meta_llama import MetaLlama2` que **no
existe como paquete real** — el propio PDF lo reconoce y remite a Hugging
Face Transformers como la vía real de implementación. Es justo lo que vamos
a usar en esta clase. Más detalles en
[manual/09_llms_gpt_llama.md](../manual/09_llms_gpt_llama.md).

## Apunte rápido: eligiendo un modelo para tu chatbot

Según el material fuente, la elección del modelo depende de la tarea:

- **Chatbots / generación de texto**: modelos seq2seq ligeros como
  `facebook/blenderbot-400M-distill` (corren en CPU) o LLM causales como
  GPT/Llama/Mistral (más potentes, más pesados).
- **Análisis de sentimiento / NER**: BERT o RoBERTa.
- **Traducción**: MarianMT, T5, mT5 o NLLB.

También hay que tener en cuenta: licencia de uso, tamaño del modelo, datos
de entrenamiento y requisitos de rendimiento. Hoy vamos a usar dos modelos
abiertos y ligeros: uno seq2seq y uno LLM causal.

## Paso 0: instalar dependencias

Usamos las mismas versiones que especifica el material fuente.

In [ ]:
!pip install transformers==4.41.2 torch==2.2.2 accelerate==0.30.1 numpy==1.26.4

print("==== Librerías instaladas =====")

## Parte 1: chatbot con un modelo seq2seq — `facebook/blenderbot-400M-distill`

Este modelo es ligero, abierto y optimizado para diálogo. Tú mismo
construyes el prompt como texto (`"User: ...\nBot:"`) y se lo pasas al
modelo en cada turno, junto con el historial reciente.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/blenderbot-400M-distill"

# Se descarga la primera vez; luego usa la copia local
model_bb = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer_bb = AutoTokenizer.from_pretrained(model_name)

print("Modelo Blenderbot cargado.")

In [ ]:
print("Chatbot (Blenderbot) listo. Escribe 'exit' para salir.\n")
conversation_history = []

while True:
    # Nos quedamos solo con los últimos intercambios
    conversation_history = conversation_history[-6:]
    history_string = "\n".join(conversation_history)

    input_text = input("> ")
    if input_text.lower() == "exit":
        break

    prompt = history_string + f"\nUser: {input_text}\nBot:"

    inputs = tokenizer_bb(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = model_bb.generate(
        **inputs,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
        repetition_penalty=1.3,
        do_sample=True,
        temperature=0.6,
        top_p=0.85
    )

    response = tokenizer_bb.decode(outputs[0], skip_special_tokens=True).strip()
    print("Bot:", response)

    conversation_history.append(f"User: {input_text}")
    conversation_history.append(f"Bot: {response}")

## Parte 2: chatbot con un LLM causal — `HuggingFaceTB/SmolLM2-360M-Instruct`

Este modelo es un LLM causal instruction-tuned: en vez de armar el prompt a
mano, usamos una lista estructurada de mensajes con roles
(`system`/`user`/`assistant`) y dejamos que
`tokenizer.apply_chat_template(...)` construya el formato exacto que el
modelo espera.

In [ ]:
from transformers import AutoModelForCausalLM
import torch
import warnings

warnings.filterwarnings("ignore")

model_name_llm = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer_llm = AutoTokenizer.from_pretrained(model_name_llm)
tokenizer_llm.pad_token = tokenizer_llm.unk_token

model_llm = AutoModelForCausalLM.from_pretrained(
    model_name_llm,
    device_map="cpu",
    torch_dtype=torch.float32
)

print("Modelo SmolLM2 cargado.")

In [ ]:
# El system prompt define la personalidad y el comportamiento del bot
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Give short and concise answers in 2-3 lines."}
]

print("Chatbot (SmolLM2) listo. Escribe 'exit' para salir.\n")
while True:
    user_input = input("> ")
    if user_input.lower() == "exit":
        break

    messages.append({"role": "user", "content": user_input})
    # Conservamos el mensaje de sistema + los últimos 10 turnos
    messages = [messages[0]] + messages[-10:]

    tokenized = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        max_length=512
    )

    with torch.inference_mode():
        outputs = model_llm.generate(
            tokenized["input_ids"],
            attention_mask=tokenized["attention_mask"],
            max_new_tokens=60,
            temperature=0.5,
            top_p=0.8,
            do_sample=True,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer_llm.pad_token_id
        )

    response = tokenizer_llm.decode(
        outputs[0][tokenized["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    print(f"Bot: {response}\n")

    messages.append({"role": "assistant", "content": response})

## Así se vería con las APIs de pago del material oficial (referencia, no ejecutable)

El PDF de Llama 2 y varios proyectos del curso (Babel Fish, chat con voz)
usan LLM de pago vía watsonx o la API de OpenAI. Por ejemplo, así se llama a
un modelo de watsonx en el proyecto Babel Fish (código real, no ejecutable
aquí porque requiere credenciales de IBM Cloud):

```python
# Así se hace con watsonx (material oficial de IBM) — requiere API key,
# NO lo ejecutes aquí. Usamos SmolLM2/Blenderbot como alternativa gratuita.
def watsonx_process_message(user_message):
    prompt = f"""
    Translate the following English sentence into Spanish.
    Reply ONLY with the translation, no explanations, no formatting, no extra text.
    English: {user_message}
    Spanish:
    """
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()
```

En esta clase usamos siempre la ruta gratuita (Hugging Face) para que puedas
ejecutar todo el código sin coste ni cuenta de pago.

## Resumen de la clase

- Un LLM se entrena con enormes cantidades de texto para generar, traducir
  o resumir lenguaje; GPT y Llama son dos familias muy conocidas.
- El código de ejemplo del PDF de Llama2 (`meta_llama.MetaLlama2`) es
  ficticio — Hugging Face Transformers es la vía real y gratuita.
- `facebook/blenderbot-400M-distill` (seq2seq) requiere construir el prompt
  a mano como texto plano.
- `HuggingFaceTB/SmolLM2-360M-Instruct` (LLM causal) usa
  `apply_chat_template` con mensajes de rol `system`/`user`/`assistant`.
- Ambos modelos corren en CPU sin necesidad de API key.

## Para seguir

- Repasa la teoría en
  [manual/09_llms_gpt_llama.md](../manual/09_llms_gpt_llama.md) y
  [manual/10_chatbot_huggingface.md](../manual/10_chatbot_huggingface.md).
- Practica lo aprendido en
  [clase06_practica_llms_chatbot_huggingface.ipynb](../notebooks_practica/clase06_practica_llms_chatbot_huggingface.ipynb),
  donde vas a cambiar el system prompt, la temperatura y el top_p del
  chatbot para darle una nueva "personalidad".